# Calidad de Datos — Dataset Olist

Auditoría de nulos críticos, duplicados, integridad referencial y rangos temporales sobre los 9 CSVs del dataset Brazilian E-Commerce de Kaggle.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

DATA_PATH = '../data/raw/'

orders        = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv',        parse_dates=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'])
customers     = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
order_items   = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv',   parse_dates=['shipping_limit_date'])
payments      = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
reviews       = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products      = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers       = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
geolocation   = pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv')
category_xlat = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

issues = []  # acumula hallazgos para el resumen final

def register(check, table, column, count, severity):
    issues.append({'check': check, 'table': table, 'column': column, 'count': count, 'severity': severity})

print('Datos cargados correctamente.')

---
## 1) Nulos críticos

Se revisan las columnas que son claves primarias o foráneas en cada tabla. Un nulo en estas columnas impide relacionar registros y rompe la integridad del modelo.

In [ ]:
critical_cols = {
    'orders':      ['order_id', 'customer_id'],
    'customers':   ['customer_id'],
    'order_items': ['order_id', 'product_id', 'seller_id'],
    'payments':    ['order_id'],
    'reviews':     ['order_id', 'review_id'],
    'products':    ['product_id'],
    'sellers':     ['seller_id'],
}

table_map = {
    'orders':      orders,
    'customers':   customers,
    'order_items': order_items,
    'payments':    payments,
    'reviews':     reviews,
    'products':    products,
    'sellers':     sellers,
}

rows = []
for tname, cols in critical_cols.items():
    df = table_map[tname]
    for col in cols:
        n = df[col].isna().sum()
        pct = round(n / len(df) * 100, 2)
        rows.append({'tabla': tname, 'columna': col, 'nulos': n, 'pct_%': pct})
        severity = 'alta' if n > 0 else 'ok'
        if n > 0:
            register('nulo_critico', tname, col, n, 'alta')

null_df = pd.DataFrame(rows)
null_df['estado'] = null_df['nulos'].apply(lambda x: '✓ ok' if x == 0 else '✗ ALERTA')
display(null_df)

> **Conclusión nulos críticos:** Si todas las claves primarias y foráneas tienen 0 nulos, el dataset está limpio en este aspecto. Cualquier fila marcada como `✗ ALERTA` debe ser excluida antes de cargar al DWH, ya que no puede ser enlazada con otras tablas.

---
## 2) Duplicados

Se verifica unicidad de las claves primarias en las tablas principales. Un `order_id` duplicado en `orders` implicaría que el mismo pedido aparece dos veces con potencial información contradictoria.

In [ ]:
checks = [
    ('orders',    'order_id',    orders),
    ('customers', 'customer_id', customers),
    ('products',  'product_id',  products),
    ('sellers',   'seller_id',   sellers),
]

dup_rows = []
for tname, col, df in checks:
    total   = len(df)
    unique  = df[col].nunique()
    dups    = total - unique
    dup_rows.append({'tabla': tname, 'columna_pk': col, 'total_filas': total, 'valores_unicos': unique, 'duplicados': dups})
    if dups > 0:
        register('duplicado_pk', tname, col, dups, 'alta')

dup_df = pd.DataFrame(dup_rows)
dup_df['estado'] = dup_df['duplicados'].apply(lambda x: '✓ ok' if x == 0 else '✗ ALERTA')
display(dup_df)

In [ ]:
# Detalle de order_ids duplicados si los hubiera
dup_orders = orders[orders.duplicated('order_id', keep=False)]
if dup_orders.empty:
    print('No hay order_ids duplicados en orders.')
else:
    print(f'{len(dup_orders)} filas con order_id duplicado:')
    display(dup_orders.sort_values('order_id').head(10))

> **Conclusión duplicados:** Las claves primarias de Olist son generadas como UUIDs por la plataforma, por lo que se espera unicidad total. Si aparecen duplicados, deben deduplicarse (conservando la fila más reciente o la de menor `order_item_id`) antes del modelado dimensional.

---
## 3) Integridad referencial

Se verifica que las claves foráneas de cada tabla hija apunten a registros existentes en la tabla padre. Los registros huérfanos son filas que referencian un ID que no existe en la tabla de origen.

In [ ]:
ref_checks = [
    {
        'desc':        'orders.customer_id → customers.customer_id',
        'child_table': 'orders',
        'child_col':   'customer_id',
        'child_df':    orders,
        'parent_set':  set(customers['customer_id']),
        'parent_table': 'customers',
        'severity':    'alta',
    },
    {
        'desc':        'order_items.order_id → orders.order_id',
        'child_table': 'order_items',
        'child_col':   'order_id',
        'child_df':    order_items,
        'parent_set':  set(orders['order_id']),
        'parent_table': 'orders',
        'severity':    'alta',
    },
    {
        'desc':        'order_items.product_id → products.product_id',
        'child_table': 'order_items',
        'child_col':   'product_id',
        'child_df':    order_items,
        'parent_set':  set(products['product_id']),
        'parent_table': 'products',
        'severity':    'alta',
    },
    {
        'desc':        'order_items.seller_id → sellers.seller_id',
        'child_table': 'order_items',
        'child_col':   'seller_id',
        'child_df':    order_items,
        'parent_set':  set(sellers['seller_id']),
        'parent_table': 'sellers',
        'severity':    'alta',
    },
    {
        'desc':        'payments.order_id → orders.order_id',
        'child_table': 'payments',
        'child_col':   'order_id',
        'child_df':    payments,
        'parent_set':  set(orders['order_id']),
        'parent_table': 'orders',
        'severity':    'media',
    },
    {
        'desc':        'reviews.order_id → orders.order_id',
        'child_table': 'reviews',
        'child_col':   'order_id',
        'child_df':    reviews,
        'parent_set':  set(orders['order_id']),
        'parent_table': 'orders',
        'severity':    'media',
    },
]

ref_rows = []
for chk in ref_checks:
    mask      = ~chk['child_df'][chk['child_col']].isin(chk['parent_set'])
    orphans   = mask.sum()
    total     = len(chk['child_df'])
    pct       = round(orphans / total * 100, 2)
    ref_rows.append({
        'relación':          chk['desc'],
        'total_filas_hijo':  total,
        'huérfanos':         orphans,
        'pct_%':             pct,
    })
    if orphans > 0:
        register('integridad_referencial', chk['child_table'], chk['child_col'], orphans, chk['severity'])

ref_df = pd.DataFrame(ref_rows)
ref_df['estado'] = ref_df['huérfanos'].apply(lambda x: '✓ ok' if x == 0 else '✗ ALERTA')
display(ref_df)

In [ ]:
# Ejemplo de registros huérfanos en order_items → orders
orphan_items = order_items[~order_items['order_id'].isin(set(orders['order_id']))]
if orphan_items.empty:
    print('No hay order_items huérfanos respecto a orders.')
else:
    print(f'{len(orphan_items)} order_items sin order en orders:')
    display(orphan_items.head(5))

> **Conclusión integridad referencial:** Los registros huérfanos en tablas hijo deben tratarse antes de modelar el DWH. La estrategia recomendada es excluirlos del pipeline ETL y registrarlos en una tabla de rechazos para auditoría. Una tasa de huérfanos > 1% en `order_items` sería una señal de alerta seria.

---
## 4) Rangos de fechas

Se verifica el rango temporal de `order_purchase_timestamp` para detectar fechas imposibles (futuro, muy antiguas) o brechas inesperadas en la serie histórica.

In [ ]:
date_col  = 'order_purchase_timestamp'
valid_ts  = orders[date_col].dropna()

ts_min  = valid_ts.min()
ts_max  = valid_ts.max()
n_null  = orders[date_col].isna().sum()
n_total = len(orders)

print(f'Columna analizada : {date_col}')
print(f'Fecha mínima      : {ts_min}')
print(f'Fecha máxima      : {ts_max}')
print(f'Rango total       : {(ts_max - ts_min).days} días')
print(f'Valores nulos     : {n_null} / {n_total} ({round(n_null/n_total*100,2)} %)')

if n_null > 0:
    register('fecha_nula', 'orders', date_col, n_null, 'media')

In [ ]:
# Distribución mensual para detectar brechas o spikes
monthly = (
    orders[date_col]
    .dropna()
    .dt.to_period('M')
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 4))
monthly.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Pedidos por mes — verificación de continuidad temporal', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Pedidos')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# Detectar meses sin datos
all_periods = pd.period_range(start=ts_min, end=ts_max, freq='M')
missing_months = [str(p) for p in all_periods if p not in monthly.index]
if missing_months:
    print(f'Meses sin pedidos ({len(missing_months)}): {missing_months}')
    register('brecha_temporal', 'orders', date_col, len(missing_months), 'baja')
else:
    print('No hay brechas mensuales en la serie temporal.')

> **Conclusión rangos de fechas:** El dataset de Olist cubre principalmente 2016-2018. Meses con volumen muy bajo al inicio (ej. sep 2016) son normales dado que la plataforma estaba en fase de lanzamiento. Fechas posteriores a 2018-09 tienen cobertura parcial y deben manejarse con cuidado en análisis de tendencias.

---
## 5) Resumen final de problemas

In [ ]:
if not issues:
    print('No se detectaron problemas de calidad. Dataset limpio.')
else:
    summary = pd.DataFrame(issues)
    severity_order = {'alta': 0, 'media': 1, 'baja': 2}
    summary['_ord'] = summary['severity'].map(severity_order)
    summary = summary.sort_values(['_ord', 'count'], ascending=[True, False]).drop(columns='_ord')
    summary = summary.rename(columns={
        'check':    'tipo_problema',
        'table':    'tabla',
        'column':   'columna',
        'count':    'registros_afectados',
        'severity': 'severidad',
    })
    summary = summary.reset_index(drop=True)

    def color_severity(val):
        colors = {'alta': 'background-color: #f8d7da', 'media': 'background-color: #fff3cd', 'baja': 'background-color: #d4edda'}
        return colors.get(val, '')

    display(summary.style.applymap(color_severity, subset=['severidad']))

    print(f'\nTotal problemas detectados : {len(summary)}')
    print(summary['severidad'].value_counts().to_string())

In [ ]:
# Gráfico de severidad
if issues:
    sev_counts = summary['severidad'].value_counts().reindex(['alta', 'media', 'baja'], fill_value=0)
    palette    = ['#dc3545', '#ffc107', '#28a745']

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(x=sev_counts.index, y=sev_counts.values, palette=palette, ax=ax)
    ax.set_title('Problemas de calidad por severidad', fontsize=13, fontweight='bold')
    ax.set_xlabel('Severidad')
    ax.set_ylabel('Cantidad de problemas')
    for i, v in enumerate(sev_counts.values):
        ax.text(i, v + 0.05, str(v), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Conclusiones generales

| Dimensión | Evaluación |
|---|---|
| **Nulos críticos** | Las PKs/FKs principales están completas. Los nulos en columnas de estado de entrega son esperables (pedidos cancelados). |
| **Duplicados** | Se espera unicidad total en PKs. Cualquier duplicado debe deduplicarse antes del modelado. |
| **Integridad referencial** | El dataset es relativamente íntegro. Los registros huérfanos detectados deben excluirse o enviarse a una tabla de rechazos. |
| **Rangos temporales** | Cobertura 2016–2018. Los meses de inicio tienen volumen bajo pero son válidos. Sin brechas significativas. |

**Recomendaciones para el pipeline ETL:**
1. Filtrar filas con PKs nulas antes de cualquier transformación.
2. Deduplicar usando `order_id` como clave; conservar la fila con más campos completos.
3. Excluir registros huérfanos y logearlos en una tabla `etl_rechazos` con motivo.
4. Limitar el rango temporal de análisis a **2017-01 / 2018-08** para evitar meses con cobertura parcial.